# Lab Sheet 6 – Building a Tool-Using AI Agent

## Part A – Building an AI Agent with Tools

## Part B – Implementing Conversation Memory Using SQLite

## Part C – Integrating RAG and Tools

## Part D – Google Calendar Integration

**Environment:** Google Colab (Python)

In [2]:
!pip -q install transformers accelerate langchain langchain-community langchain-huggingface langchain-chroma sentence-transformers pypdf chromadb psutil

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("AI model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

AI model loaded successfully!


In [4]:
messages = [
    {"role": "user", "content": "What is Artificial Intelligence?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("AI Response:")
print(response)

AI Response:
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves developing algorithms and models that can process information, learn from data, and make decisions based on that learning.

Key features of AI include:

1. **Learning**: The ability to improve its performance over time through experience.
2. **Reasoning**: The capacity to understand and use language effectively.
3. **Perception**: The capability to perceive and respond to stimuli in


In [5]:
import ast
import operator
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression using +, -, *, / and **."""

    operations = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg
    }

    def evaluate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in operations:
            return operations[type(node.op)](
                evaluate(node.left),
                evaluate(node.right)
            )

        if isinstance(node, ast.UnaryOp) and type(node.op) in operations:
            return operations[type(node.op)](evaluate(node.operand))

        raise ValueError("Unsupported expression")

    try:
        tree = ast.parse(expression, mode="eval")
        result = evaluate(tree.body)
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"

print("Calculator tool created successfully!")
print("Test 1:", calculator.invoke("20 + 20"))
print("Test 2:", calculator.invoke("10 * 5"))
print("Test 3:", calculator.invoke("100 / 4"))

Calculator tool created successfully!
Test 1: 40
Test 2: 50
Test 3: 25.0


In [6]:
from datetime import datetime
import platform
import psutil
import random
from langchain_core.tools import tool

@tool
def current_datetime() -> str:
    """Get the current date and time."""
    return str(datetime.now())

@tool
def system_information() -> str:
    """Get information about the operating system."""
    return platform.platform()

@tool
def memory_usage() -> str:
    """Get current system RAM usage."""
    memory = psutil.virtual_memory()
    return f"RAM Usage: {memory.percent}%"

@tool
def random_number() -> str:
    """Generate a random number between 1 and 100."""
    return str(random.randint(1, 100))

@tool
def word_count(text: str) -> str:
    """Count the number of words in a given text."""
    return str(len(text.split()))

print("All five tools created successfully!")

print("Date and Time:", current_datetime.invoke({}))
print("System Information:", system_information.invoke({}))
print("Memory Usage:", memory_usage.invoke({}))
print("Random Number:", random_number.invoke({}))
print("Word Count:", word_count.invoke("I am learning Artificial Intelligence"))

All five tools created successfully!
Date and Time: 2026-11-23 13:11:31.498039
System Information: Linux-6.6.159+-x86_64-with-glibc2.39
Memory Usage: RAM Usage: 42.7%
Random Number: 23
Word Count: 5


In [7]:
# Register all six tools
tools = [
    calculator,
    current_datetime,
    system_information,
    memory_usage,
    random_number,
    word_count
]

# Create a tool lookup dictionary
tool_map = {
    tool.name: tool
    for tool in tools
}

print("All tools registered successfully!")

for tool in tools:
    print("-", tool.name)

All tools registered successfully!
- calculator
- current_datetime
- system_information
- memory_usage
- random_number
- word_count


In [8]:
# Give the AI model information about our tools
tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.args_schema.model_json_schema()
        }
    }
    for t in tools
]

def ask_agent(question):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Use the available tools when appropriate. "
                "Do not invent tool results."
            )
        },
        {"role": "user", "content": question}
    ]

    for _ in range(3):
        prompt = tokenizer.apply_chat_template(
            messages,
            tools=tool_definitions,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        # Check whether the model requested a tool
        import re
        import json

        matches = re.findall(
            r"<tool_call>\s*(.*?)\s*</tool_call>",
            response,
            flags=re.DOTALL
        )

        if not matches:
            return response

        messages.append({
            "role": "assistant",
            "content": response
        })

        for match in matches:
            try:
                call = json.loads(match)
                name = call["name"]
                arguments = call.get("arguments", {})

                if name not in tool_map:
                    result = f"Unknown tool: {name}"
                else:
                    print("Selected Tool:", name)
                    result = tool_map[name].invoke(arguments)

            except Exception as e:
                result = f"Tool execution error: {e}"

            messages.append({
                "role": "tool",
                "content": str(result)
            })

    return "The agent reached its tool-use limit."


print("AI Agent created successfully!")

AI Agent created successfully!


In [9]:
print("TEST 1: Calculator")
print(ask_agent("What is 25 multiplied by 4?"))

print("\nTEST 2: Date and Time")
print(ask_agent("What is the current date and time?"))

print("\nTEST 3: Word Counter")
print(ask_agent("Count the words in: I am studying Artificial Intelligence"))


TEST 1: Calculator
Selected Tool: calculator
The result of multiplying 25 by 4 is 100.

TEST 2: Date and Time
Selected Tool: current_datetime
The current date and time is November 23, 2026, at 1:12 PM.

TEST 3: Word Counter
Selected Tool: word_count
The word count for your text is 5.


# Part B – Implementing Conversation Memory Using SQLite

This section implements conversation memory using SQLite to store and retrieve previous interactions between the user and the AI agent.

In [10]:
import sqlite3

# Connect to SQLite database
conn = sqlite3.connect("agent_memory.db")
cursor = conn.cursor()

# Create conversation history table
cursor.execute("""
CREATE TABLE IF NOT EXISTS conversation_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_question TEXT,
    agent_answer TEXT
)
""")

conn.commit()

print("Conversation memory database created successfully!")

Conversation memory database created successfully!


In [11]:
# Function to save conversations
def save_conversation(question, answer):
    cursor.execute("""
        INSERT INTO conversation_history (user_question, agent_answer)
        VALUES (?, ?)
    """, (question, answer))

    conn.commit()
    return "Conversation saved successfully!"


# Test saving a conversation
result = save_conversation(
    "What is my favourite subject?",
    "Your favourite subject is Data Science."
)

print(result)

Conversation saved successfully!


In [12]:
# Function to retrieve previous conversations
def get_conversation_history():
    cursor.execute("""
        SELECT user_question, agent_answer
        FROM conversation_history
        ORDER BY id DESC
    """)

    return cursor.fetchall()


# Test retrieving conversations
history = get_conversation_history()

for question, answer in history:
    print("User:", question)
    print("AI Agent:", answer)
    print("------------------------")

User: What is my favourite subject?
AI Agent: Your favourite subject is Data Science.
------------------------


In [17]:
import re

def search_memory(question):

    cursor.execute("""
        SELECT user_question, agent_answer
        FROM conversation_history
        ORDER BY id DESC
    """)

    records = cursor.fetchall()

    if not records:
        return "No conversation history found."

    # Ignore common question words
    stop_words = {
        "what", "when", "where", "which",
        "who", "how", "does", "have",
        "your", "with", "this", "that"
    }

    def keywords(text):
        words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
        return {
            word for word in words
            if len(word) > 3 and word not in stop_words
        }

    question_keywords = keywords(question)

    for previous_question, previous_answer in records:

        previous_keywords = keywords(previous_question)

        if question_keywords and question_keywords == previous_keywords:
            return previous_answer

    return "No matching conversation found."


print("Memory search updated successfully!")

Memory search updated successfully!


In [14]:
# Keep the original AI agent from Part A
original_ask_agent = ask_agent

# Create a memory-enabled agent
def ask_agent_with_memory(question):

    # Check previous conversations
    memory_result = search_memory(question)

    if memory_result != "No conversation history found." and \
       memory_result != "No matching conversation found.":

        print("Selected Tool: Conversation Memory")
        return memory_result

    # If no matching memory, use the AI model
    response = original_ask_agent(question)

    # Save the new conversation
    save_conversation(question, response)

    return response


print("Memory-enabled AI Agent created successfully!")

Memory-enabled AI Agent created successfully!


In [15]:
print("TEST 1: Remembering a previous conversation")

response = ask_agent_with_memory(
    "What is my favourite subject?"
)

print("AI Response:", response)

TEST 1: Remembering a previous conversation
Selected Tool: Conversation Memory
AI Response: Your favourite subject is Data Science.


In [18]:
# Test a new conversation
print("TEST 2: New conversation")

response = ask_agent_with_memory(
    "What is 15 multiplied by 3?"
)

print("AI Response:", response)

# Check the saved conversations
print("\nSaved Conversation History:")

history = get_conversation_history()

for question, answer in history:
    print("User:", question)
    print("AI Agent:", answer)
    print("------------------------")

TEST 2: New conversation
Selected Tool: calculator
AI Response: The result of multiplying 15 by 3 is 45.

Saved Conversation History:
User: What is 15 multiplied by 3?
AI Agent: The result of multiplying 15 by 3 is 45.
------------------------
User: What is my favourite subject?
AI Agent: Your favourite subject is Data Science.
------------------------


# Part C – Integrating RAG and Tools

This section integrates Retrieval-Augmented Generation (RAG) with the AI agent.

The agent will use three main tools:

1. Calculator Tool
2. Memory Search Tool
3. RAG Search Tool

The RAG system will retrieve relevant information from PDF documents to help the AI agent answer questions.

In [20]:
from google.colab import files

# Upload PDF documents
uploaded = files.upload()

print("PDF files uploaded successfully!")
print("Uploaded files:", list(uploaded.keys()))

Saving Lab sheet 6.pdf to Lab sheet 6.pdf
PDF files uploaded successfully!
Uploaded files: ['Lab sheet 6.pdf']


In [21]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the uploaded PDF
pdf_path = "Lab sheet 6.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total PDF pages loaded:", len(documents))

# Split PDF content into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total text chunks created:", len(chunks))
print("PDF processing completed successfully!")

/tmp/ipykernel_328/3023165451.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total PDF pages loaded: 12
Total text chunks created: 12
PDF processing completed successfully!


In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Create the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store PDF chunks in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="labsheet6_rag"
)

print("RAG vector database created successfully!")
print("Total documents stored:", vectorstore._collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG vector database created successfully!
Total documents stored: 12


In [23]:
# Test retrieving information from the PDF
query = "What are the requirements for Part C?"

results = vectorstore.similarity_search(
    query,
    k=3
)

print("RAG Search Results:\n")

for i, document in enumerate(results, start=1):
    print(f"Result {i}:")
    print(document.page_content[:700])
    print("------------------------")

RAG Search Results:

Result 1:
PUSL 3205-Large Language Models and Generative AI 
 
Exercise-Expand with Multiple tools 
1. Add multiple tools 
• Date and Time – retrieves the current date and time. 
• Notepad – opens Windows Notepad and writes user-provided text. 
• File Search – searches for files by name or extension in a specified directory. 
• File Creation – creates a text file with user-provided content. 
• System Information – retrieves operating system, processor and memory information. 
 
2. Test all the tools 
No. User query  
1 Calculate 25 × 4  
2 What is the date today? 
3 Open Notepad and write Hello Students  
4 Find all PDF files in D:\MSC AI  
5 Create a file called test.txt in D:\MSC AI with Hello World
------------------------
Result 2:
PUSL 3205-Large Language Models and Generative AI 
 
Part C-RAG powered AI Agent 
The agent will have access to three tools: 
• Calculator Tool: Performs mathematical calculations. 
• Memory Search Tool: Retrieves previously stored u

In [24]:
from langchain_core.tools import tool

@tool
def rag_search(query: str) -> str:
    """Search the university PDF knowledge base for relevant information."""

    results = vectorstore.similarity_search(query, k=3)

    if not results:
        return "No relevant information found in the PDF."

    context = "\n\n".join(
        document.page_content
        for document in results
    )

    return context


print("RAG Search Tool created successfully!")

# Test the tool
result = rag_search.invoke({
    "query": "What are the three tools required in Part C?"
})

print("\nRAG Search Result:")
print(result[:1000])

RAG Search Tool created successfully!

RAG Search Result:
PUSL 3205-Large Language Models and Generative AI 
 
Part C-RAG powered AI Agent 
The agent will have access to three tools: 
• Calculator Tool: Performs mathematical calculations. 
• Memory Search Tool: Retrieves previously stored user information. 
• RAG Search Tool: Retrieves relevant information from a university knowledge base. 
[Hint: Use the RAG we implemented Lab session 5]

PUSL 3205-Large Language Models and Generative AI 
 
Exercise-Expand with Multiple tools 
1. Add multiple tools 
• Date and Time – retrieves the current date and time. 
• Notepad – opens Windows Notepad and writes user-provided text. 
• File Search – searches for files by name or extension in a specified directory. 
• File Creation – creates a text file with user-provided content. 
• System Information – retrieves operating system, processor and memory information. 
 
2. Test all the tools 
No. User query  
1 Calculate 25 × 4  
2 What is the date tod

In [25]:
from langchain_core.tools import tool

@tool
def memory_search(query: str) -> str:
    """Search previous conversations stored in SQLite memory."""

    result = search_memory(query)

    return result


print("Memory Search Tool created successfully!")

# Test the tool
result = memory_search.invoke({
    "query": "What is my favourite subject?"
})

print("Memory Search Result:", result)

Memory Search Tool created successfully!
Memory Search Result: Your favourite subject is Data Science.


In [26]:
# Register the three Part C tools
part_c_tools = [
    calculator,
    memory_search,
    rag_search
]

# Create tool lookup dictionary
part_c_tool_map = {
    t.name: t
    for t in part_c_tools
}

# Create tool definitions for Qwen
part_c_tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.args_schema.model_json_schema()
        }
    }
    for t in part_c_tools
]

print("Part C tools registered successfully!")

for t in part_c_tools:
    print("-", t.name)

Part C tools registered successfully!
- calculator
- memory_search
- rag_search


In [32]:
import re

def part_c_agent(question):

    question_lower = question.lower()

    # 1. Select Calculator Tool
    math_words = [
        "calculate", "multiply", "multiplied",
        "divide", "divided", "plus", "minus",
        "add", "subtract"
    ]

    if any(word in question_lower for word in math_words):

        print("Selected Tool: calculator")

        expression = question_lower

        expression = expression.replace("multiplied by", "*")
        expression = expression.replace("multiply", "*")
        expression = expression.replace("times", "*")
        expression = expression.replace("divided by", "/")
        expression = expression.replace("plus", "+")
        expression = expression.replace("minus", "-")

        match = re.search(
            r"-?\d+(?:\.\d+)?\s*[\+\-\*/]\s*-?\d+(?:\.\d+)?",
            expression
        )

        if match:
            result = calculator.invoke(match.group())
            return f"The calculation result is {result}."

        return "Please provide a valid mathematical expression."

    # 2. Select Memory Search Tool
    memory_words = [
        "my favourite", "my favorite",
        "remember", "previous conversation",
        "what did i", "my preference"
    ]

    if any(word in question_lower for word in memory_words):

        print("Selected Tool: memory_search")

        result = memory_search.invoke({
            "query": question
        })

        return result

    # 3. Select RAG Search Tool
    rag_words = [
        "lab sheet", "labsheet",
        "part c", "pdf", "document",
        "coursework", "assignment"
    ]

    if any(word in question_lower for word in rag_words):

        print("Selected Tool: rag_search")

        result = rag_search.invoke({
            "query": question
        })

        return result[:1500]

    # 4. Use AI model for general questions
    return ask_agent(question)


print("Part C AI Agent updated successfully!")

Part C AI Agent updated successfully!


In [33]:
print("TEST 1: Calculator Tool")
print(part_c_agent("Calculate 25 multiplied by 4"))

print("\nTEST 2: Memory Search Tool")
print(part_c_agent("What is my favourite subject?"))

print("\nTEST 3: RAG Search Tool")
print(part_c_agent(
    "According to Lab Sheet 6, what are the three tools required in Part C?"
))

TEST 1: Calculator Tool
Selected Tool: calculator
The calculation result is 100.

TEST 2: Memory Search Tool
Selected Tool: memory_search
Your favourite subject is Data Science.

TEST 3: RAG Search Tool
Selected Tool: rag_search
PUSL 3205-Large Language Models and Generative AI 
 
Part C-RAG powered AI Agent 
The agent will have access to three tools: 
• Calculator Tool: Performs mathematical calculations. 
• Memory Search Tool: Retrieves previously stored user information. 
• RAG Search Tool: Retrieves relevant information from a university knowledge base. 
[Hint: Use the RAG we implemented Lab session 5]

PUSL 3205-Large Language Models and Generative AI 
 
Exercise-Expand with Multiple tools 
1. Add multiple tools 
• Date and Time – retrieves the current date and time. 
• Notepad – opens Windows Notepad and writes user-provided text. 
• File Search – searches for files by name or extension in a specified directory. 
• File Creation – creates a text file with user-provided content. 


# Part D – Google Calendar Integration

This section integrates Google Calendar with the AI agent.

The system will be able to:

1. Retrieve upcoming calendar events.
2. Check scheduled events.
3. Create new calendar events with user confirmation.

In [34]:
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [35]:
from google.colab import files

uploaded_credentials = files.upload()

credentials_filename = list(uploaded_credentials.keys())[0]

print("Credentials file uploaded successfully!")

Saving client_secret_532818014687-k5ojc03kbg85dip1lh1ut0urdf4vq79p.apps.googleusercontent.com.json to client_secret_532818014687-k5ojc03kbg85dip1lh1ut0urdf4vq79p.apps.googleusercontent.com.json
Credentials file uploaded successfully!


In [37]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# Google Calendar permissions
SCOPES = [
    "https://www.googleapis.com/auth/calendar.events"
]

# Load the uploaded OAuth credentials
flow = InstalledAppFlow.from_client_secrets_file(
    credentials_filename,
    scopes=SCOPES
)

flow.redirect_uri = "http://localhost"

# Generate authorization URL
authorization_url, state = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

print("OAuth authorization URL generated.")
print("Open the authorization URL privately to approve access.")

OAuth authorization URL generated.
Open the authorization URL privately to approve access.


In [38]:
from IPython.display import display, HTML

display(HTML(
    '<a href="' + authorization_url + '" target="_blank">'
    'Click here to authorize Google Calendar'
    '</a>'
))

In [39]:
# Generate a fresh Google authorization link
authorization_url, state = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

from IPython.display import display, HTML
from html import escape

display(HTML(
    '<a href="' + escape(authorization_url, quote=True) +
    '" target="_blank">Authorize Google Calendar</a>'
))

print("New authorization link generated successfully!")

New authorization link generated successfully!


In [41]:
import os
from getpass import getpass
from googleapiclient.discovery import build

# Allow the localhost OAuth redirect
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

try:
    redirected_url = getpass("Paste the full localhost URL here: ")

    flow.fetch_token(authorization_response=redirected_url)

    service = build(
        "calendar",
        "v3",
        credentials=flow.credentials
    )

    print("Google Calendar connected successfully!")

finally:
    if "redirected_url" in locals():
        del redirected_url
    os.environ.pop("OAUTHLIB_INSECURE_TRANSPORT", None)

Paste the full localhost URL here: ··········
Google Calendar connected successfully!


In [42]:
from datetime import datetime, timezone

# Get the current date and time
now = datetime.now(timezone.utc).isoformat()

# Retrieve the next 10 calendar events
events_result = service.events().list(
    calendarId="primary",
    timeMin=now,
    maxResults=10,
    singleEvents=True,
    orderBy="startTime"
).execute()

events = events_result.get("items", [])

print("Upcoming Google Calendar Events:")

if not events:
    print("No upcoming events found.")
else:
    for event in events:
        start = event["start"].get(
            "dateTime",
            event["start"].get("date")
        )

        print(f"{start} - {event.get('summary', 'Untitled Event')}")

Upcoming Google Calendar Events:
2026-12-04 - Happy birthday!
2027-12-04 - Happy birthday!
2028-12-04 - Happy birthday!
2029-12-04 - Happy birthday!
2030-12-04 - Happy birthday!
2031-12-04 - Happy birthday!
2032-12-04 - Happy birthday!
2033-12-04 - Happy birthday!
2034-12-04 - Happy birthday!
2035-12-04 - Happy birthday!


In [43]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

def check_today_schedule():
    timezone = ZoneInfo("Asia/Colombo")

    today = datetime.now(timezone)
    start = today.replace(hour=0, minute=0, second=0, microsecond=0)
    end = start + timedelta(days=1)

    result = service.events().list(
        calendarId="primary",
        timeMin=start.isoformat(),
        timeMax=end.isoformat(),
        singleEvents=True,
        orderBy="startTime"
    ).execute()

    events = result.get("items", [])

    if not events:
        return "No events scheduled for today."

    schedule = "Today's Schedule:\n"

    for event in events:
        event_time = event["start"].get(
            "dateTime",
            event["start"].get("date")
        )

        schedule += f"{event_time} - {event.get('summary', 'Untitled Event')}\n"

    return schedule


print(check_today_schedule())

Today's Schedule:
2026-11-23T10:00:00+05:30 - AI Agent Integration Test



In [44]:
from datetime import datetime
from zoneinfo import ZoneInfo

def create_calendar_event(title, start_time, end_time):

    timezone = ZoneInfo("Asia/Colombo")

    start = datetime.fromisoformat(start_time).replace(tzinfo=timezone)
    end = datetime.fromisoformat(end_time).replace(tzinfo=timezone)

    if end <= start:
        return "Error: End time must be after start time."

    print("Event Details:")
    print("Title:", title)
    print("Start:", start)
    print("End:", end)

    confirmation = input(
        "Do you want to create this event? (yes/no): "
    )

    if confirmation.lower().strip() != "yes":
        return "Event creation cancelled."

    event = {
        "summary": title,
        "start": {
            "dateTime": start.isoformat(),
            "timeZone": "Asia/Colombo"
        },
        "end": {
            "dateTime": end.isoformat(),
            "timeZone": "Asia/Colombo"
        }
    }

    created_event = service.events().insert(
        calendarId="primary",
        body=event
    ).execute()

    return "Event created successfully! ID: " + created_event["id"]


print("Calendar event creation function ready!")

Calendar event creation function ready!


In [45]:
# Test Google Calendar event creation

result = create_calendar_event(
    title="Lab Sheet 6 AI Agent Test",
    start_time="2026-11-25T10:00:00",
    end_time="2026-11-25T11:00:00"
)

print(result)

Event Details:
Title: Lab Sheet 6 AI Agent Test
Start: 2026-11-25 10:00:00+05:30
End: 2026-11-25 11:00:00+05:30
Do you want to create this event? (yes/no): yes
Event created successfully! ID: i0t0s8lk18n08sdqaag1o8pkhk


In [46]:
# Save the existing Part C agent
previous_part_c_agent = part_c_agent

def university_agent(question, event_details=None):

    question_lower = question.lower()

    # Check today's calendar
    if any(phrase in question_lower for phrase in [
        "today's schedule",
        "my schedule today",
        "calendar today",
        "events today"
    ]):
        print("Selected Tool: Google Calendar")
        return check_today_schedule()

    # Create a calendar event
    if any(phrase in question_lower for phrase in [
        "create event",
        "create an event",
        "schedule a meeting",
        "add event"
    ]):
        print("Selected Tool: Create Calendar Event")

        if event_details is None:
            return "Please provide the event title, start time, and end time."

        return create_calendar_event(
            title=event_details["title"],
            start_time=event_details["start_time"],
            end_time=event_details["end_time"]
        )

    # Use the existing Part C agent for other questions
    return previous_part_c_agent(question)


print("Google Calendar integrated with AI Agent successfully!")


Google Calendar integrated with AI Agent successfully!


In [47]:
# TEST 1: Google Calendar
print("TEST 1: Today's Schedule")
print(university_agent("What is my schedule today?"))

print("\n-------------------------\n")

# TEST 2: Calculator
print("TEST 2: Calculator")
print(university_agent("Calculate 25 multiplied by 4"))

print("\n-------------------------\n")

# TEST 3: Conversation Memory
print("TEST 3: Memory Search")
print(university_agent("What is my favourite subject?"))

print("\n-------------------------\n")

# TEST 4: RAG Search
print("TEST 4: RAG Search")
print(university_agent(
    "What are the three tools required in Part C of Lab Sheet 6?"
))

TEST 1: Today's Schedule
Selected Tool: Google Calendar
Today's Schedule:
2026-11-23T10:00:00+05:30 - AI Agent Integration Test


-------------------------

TEST 2: Calculator
Selected Tool: calculator
The calculation result is 100.

-------------------------

TEST 3: Memory Search
Selected Tool: memory_search
Your favourite subject is Data Science.

-------------------------

TEST 4: RAG Search
Selected Tool: rag_search
PUSL 3205-Large Language Models and Generative AI 
 
Part C-RAG powered AI Agent 
The agent will have access to three tools: 
• Calculator Tool: Performs mathematical calculations. 
• Memory Search Tool: Retrieves previously stored user information. 
• RAG Search Tool: Retrieves relevant information from a university knowledge base. 
[Hint: Use the RAG we implemented Lab session 5]

PUSL 3205-Large Language Models and Generative AI 
 
Exercise-Expand with Multiple tools 
1. Add multiple tools 
• Date and Time – retrieves the current date and time. 
• Notepad – opens Wi

# Final Testing and Results

## Lab Sheet 6 – Building a Tool-Using AI Agent

The AI agent was implemented and tested using Google Colab.

### Part A – Basic AI Agent
- Loaded a language model.
- Implemented calculator and utility tools.
- Tested mathematical calculations, date and time, and word counting.

### Part B – Conversation Memory
- Created an SQLite database.
- Stored previous user conversations.
- Retrieved information from conversation history.

### Part C – RAG-Powered AI Agent
- Uploaded a university PDF document.
- Processed and divided the document into chunks.
- Created a vector database using ChromaDB.
- Integrated Calculator, Memory Search, and RAG Search tools.
- Successfully tested all three tools.

### Part D – Google Calendar Integration
- Connected Google Calendar using OAuth authentication.
- Retrieved upcoming calendar events.
- Checked today's schedule.
- Created a calendar event after user confirmation.
- Integrated Google Calendar with the existing AI agent.

## Final Results

All four integrated tests completed successfully:
1. Google Calendar – Passed
2. Calculator – Passed
3. Memory Search – Passed
4. RAG Search – Passed

## Implementation Note

The system uses Python keyword-based routing to select tools.
A small language model is used for general AI responses.
The system was developed in Google Colab rather than using a locally installed Ollama model.